<a href="https://colab.research.google.com/github/WilliamCenturion/Automacao-GSheets-Discord/blob/main/VEXA_Signals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import requests
import pandas as pd

URL = "https://api.hyperliquid.xyz/info"

response = requests.post(
    URL,
    json={"type": "metaAndAssetCtxs"},
    timeout=20
)

response.raise_for_status()

data = response.json()

universe = data[0]["universe"]
contexts = data[1]

markets = []

for coin, ctx in zip(universe, contexts):
    markets.append({
        "Moeda": coin["name"],
        "Preço": float(ctx["markPx"]),
        "Volume_24h": float(ctx["dayNtlVlm"]),
        "Funding": float(ctx["funding"])
    })

df = pd.DataFrame(markets)

df = df.sort_values("Volume_24h", ascending=False)

print("VEXA SIGNALS — CONEXÃO ESTABELECIDA")
print(f"Mercados encontrados: {len(df)}")
display(df.head(15))


In [ ]:

# VEXA SIGNALS — FILTRO DE MERCADOS

MIN_VOLUME = 30_000_000

df["Elegível"] = df["Volume_24h"] >= MIN_VOLUME

df_eligible = df[df["Elegível"]].copy()

df_eligible["Grupo"] = df_eligible["Preço"].apply(
    lambda x: "Abaixo de $1" if x < 1 else "Alta liquidez"
)

print("VEXA — MERCADOS ELEGÍVEIS")
print(f"Total de ativos: {len(df_eligible)}")

display(
    df_eligible[
        ["Moeda", "Preço", "Volume_24h", "Funding", "Grupo"]
    ].head(30)
)


In [ ]:

# ==========================================
# VEXA SIGNALS — MOTOR DE INDICADORES
# ==========================================

import time
import numpy as np
from datetime import datetime, timezone

def get_candles(coin, interval="15m", limit=260):
    now = int(datetime.now(timezone.utc).timestamp() * 1000)

    payload = {
        "type": "candleSnapshot",
        "req": {
            "coin": coin,
            "interval": interval,
            "startTime": now - limit * 15 * 60 * 1000,
            "endTime": now
        }
    }

    r = requests.post(URL, json=payload, timeout=20)
    r.raise_for_status()

    candles = r.json()

    df_c = pd.DataFrame(candles)

    if df_c.empty:
        return df_c

    for col in ["o", "h", "l", "c", "v"]:
        df_c[col] = df_c[col].astype(float)

    # Manter somente candles encerrados
    df_c = df_c[df_c["T"].astype(int) <= now].copy()

    return df_c.sort_values("t").reset_index(drop=True)


def calculate_indicators(d):
    d = d.copy()

    close = d["c"]
    high = d["h"]
    low = d["l"]

    # Médias exponenciais
    for period in [9, 21, 50, 200]:
        d[f"EMA{period}"] = close.ewm(
            span=period, adjust=False
        ).mean()

    # RSI 14
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    avg_loss = loss.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    d["RSI"] = 100 - (100 / (1 + rs))
    d.loc[avg_loss == 0, "RSI"] = 100

    # ADX 14
    up_move = high.diff()
    down_move = -low.diff()

    plus_dm = up_move.where(
        (up_move > down_move) & (up_move > 0), 0.0
    )

    minus_dm = down_move.where(
        (down_move > up_move) & (down_move > 0), 0.0
    )

    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low - close.shift()).abs()
    ], axis=1).max(axis=1)

    atr = tr.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    plus_di = 100 * plus_dm.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean() / atr

    minus_di = 100 * minus_dm.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean() / atr

    dx = 100 * (plus_di - minus_di).abs() / (
        plus_di + minus_di
    ).replace(0, np.nan)

    d["ADX"] = dx.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    # Volume relativo à média das 20 velas anteriores
    d["VOL_MEDIA"] = d["v"].shift(1).rolling(20).mean()
    d["VOL_RATIO"] = d["v"] / d["VOL_MEDIA"]

    return d


# Analisar os mercados selecionados
results = []

for coin in df_eligible["Moeda"]:
    try:
        candles = get_candles(coin)

        if len(candles) < 220:
            print(f"Histórico insuficiente: {coin}")
            continue

        candles = calculate_indicators(candles)
        last = candles.iloc[-1]

        required = [
            "EMA9", "EMA21", "EMA50", "EMA200",
            "RSI", "ADX", "VOL_RATIO"
        ]

        if last[required].isna().any():
            continue

        long_trend = (
            last["EMA9"] > last["EMA21"] >
            last["EMA50"] > last["EMA200"]
        )

        short_trend = (
            last["EMA9"] < last["EMA21"] <
            last["EMA50"] < last["EMA200"]
        )

        long_signal = (
            long_trend
            and last["c"] > last["EMA9"]
            and 52 <= last["RSI"] <= 70
            and last["ADX"] >= 20
            and last["VOL_RATIO"] >= 1.2
        )

        short_signal = (
            short_trend
            and last["c"] < last["EMA9"]
            and 30 <= last["RSI"] <= 48
            and last["ADX"] >= 20
            and last["VOL_RATIO"] >= 1.2
        )

        signal = (
            "LONG" if long_signal
            else "SHORT" if short_signal
            else "AGUARDAR"
        )

        results.append({
            "Moeda": coin,
            "Preço": last["c"],
            "RSI": round(last["RSI"], 2),
            "ADX": round(last["ADX"], 2),
            "Volume x Média": round(last["VOL_RATIO"], 2),
            "Tendência": (
                "ALTA" if long_trend
                else "BAIXA" if short_trend
                else "NEUTRA"
            ),
            "Sinal": signal
        })

    except Exception as e:
        print(f"Erro em {coin}: {e}")

    time.sleep(0.15)


signals_df = pd.DataFrame(results)

if not signals_df.empty:
    signals_df = signals_df.sort_values(
        "Sinal",
        key=lambda s: s.map({
            "LONG": 0,
            "SHORT": 1,
            "AGUARDAR": 2
        })
    )

    print("VEXA SIGNALS — ANÁLISE CONCLUÍDA")
    print(f"Ativos analisados: {len(signals_df)}")
    print(f"LONG: {(signals_df['Sinal'] == 'LONG').sum()}")
    print(f"SHORT: {(signals_df['Sinal'] == 'SHORT').sum()}")

    display(signals_df)
else:
    print("Nenhum resultado disponível.")


In [ ]:

# ==========================================
# VEXA SIGNALS — CONFLUENCE SCORE
# ==========================================

from datetime import datetime, timezone

def calculate_score(row):
    scores = {"LONG": 0, "SHORT": 0}
    reasons = {"LONG": [], "SHORT": []}

    # 1. Tendência: até 35 pontos
    if row["Tendência"] == "ALTA":
        scores["LONG"] += 35
        reasons["LONG"].append("EMAs alinhadas para alta")

    elif row["Tendência"] == "BAIXA":
        scores["SHORT"] += 35
        reasons["SHORT"].append("EMAs alinhadas para baixa")

    # 2. RSI: até 20 pontos
    rsi = row["RSI"]

    if 52 <= rsi <= 70:
        scores["LONG"] += 20
        reasons["LONG"].append("RSI favorável para LONG")

    elif 30 <= rsi <= 48:
        scores["SHORT"] += 20
        reasons["SHORT"].append("RSI favorável para SHORT")

    # 3. ADX: até 25 pontos
    adx = row["ADX"]
    adx_score = min(25, max(0, (adx / 30) * 25))

    if adx >= 20:
        scores["LONG"] += adx_score
        scores["SHORT"] += adx_score
        reasons["LONG"].append("Força de tendência confirmada")
        reasons["SHORT"].append("Força de tendência confirmada")

    # 4. Volume: até 20 pontos
    volume = row["Volume x Média"]
    volume_score = min(20, max(0, (volume / 2) * 20))

    if volume >= 1.2:
        scores["LONG"] += volume_score
        scores["SHORT"] += volume_score
        reasons["LONG"].append("Volume acima da média")
        reasons["SHORT"].append("Volume acima da média")

    direction = row["Sinal"]

    if direction == "LONG":
        selected_score = scores["LONG"]
        explanation = reasons["LONG"]

    elif direction == "SHORT":
        selected_score = scores["SHORT"]
        explanation = reasons["SHORT"]

    else:
        selected_score = max(scores.values())
        explanation = (
            reasons["LONG"]
            if scores["LONG"] >= scores["SHORT"]
            else reasons["SHORT"]
        )

    return pd.Series({
        "Score": round(selected_score),
        "Justificativa": " | ".join(explanation)
    })


# Criar nova versão do painel
vexa_panel = signals_df.copy()

score_data = vexa_panel.apply(calculate_score, axis=1)

vexa_panel = pd.concat(
    [vexa_panel, score_data], axis=1
)

vexa_panel["Referência"] = vexa_panel["Preço"]
vexa_panel["Horário da análise"] = datetime.now(
    timezone.utc
).strftime("%H:%M:%S UTC")

# Ordenar por sinal e score
vexa_panel = vexa_panel.sort_values(
    "Score", ascending=False
)

# Painel final
print("===================================")
print("          VEXA SIGNALS")
print("===================================")

print("LONG:", (vexa_panel["Sinal"] == "LONG").sum())
print("SHORT:", (vexa_panel["Sinal"] == "SHORT").sum())
print("AGUARDAR:", (vexa_panel["Sinal"] == "AGUARDAR").sum())

print("\nRANKING DE OPORTUNIDADES")

display(
    vexa_panel[
        [
            "Moeda",
            "Sinal",
            "Score",
            "Preço",
            "RSI",
            "ADX",
            "Volume x Média",
            "Justificativa"
        ]
    ].reset_index(drop=True)
)


In [ ]:

# VEXA SIGNALS — RANKING INTELIGENTE

vexa_panel["Prioridade"] = vexa_panel["Sinal"].map({
    "LONG": 0,
    "SHORT": 0,
    "AGUARDAR": 1
})

vexa_panel = vexa_panel.sort_values(
    ["Prioridade", "Score"],
    ascending=[True, False]
).reset_index(drop=True)

print("========== VEXA SIGNALS ==========")
print("SINAIS LONG:",
      (vexa_panel["Sinal"] == "LONG").sum())

print("SINAIS SHORT:",
      (vexa_panel["Sinal"] == "SHORT").sum())

print("EM ESPERA:",
      (vexa_panel["Sinal"] == "AGUARDAR").sum())

display(
    vexa_panel[
        ["Moeda", "Sinal", "Score", "Preço",
         "RSI", "ADX", "Volume x Média"]
    ]
)


In [ ]:

from IPython.display import display, HTML

longs = (vexa_panel["Sinal"] == "LONG").sum()
shorts = (vexa_panel["Sinal"] == "SHORT").sum()
waiting = (vexa_panel["Sinal"] == "AGUARDAR").sum()

table = vexa_panel[
    ["Moeda", "Sinal", "Score", "Preço",
     "RSI", "ADX", "Volume x Média"]
].copy()

table["Score"] = table["Score"].astype(str) + "/100"

html_table = table.to_html(
    index=False, border=0, classes="vexa-table"
)

html = f"""
<div style="background:#080808;color:#eeeeee;
padding:28px;border-radius:14px;font-family:Arial,sans-serif">

<div style="display:flex;justify-content:space-between;align-items:center">
<h1 style="margin:0;color:#F5C518;letter-spacing:3px">VEXA</h1>
<span style="color:#F5C518;font-weight:bold">SIGNALS</span>
</div>

<p style="color:#888;font-size:11px;letter-spacing:2px">
CRYPTO MARKET INTELLIGENCE
</p>

<hr style="border:1px solid #F5C518">

<div style="display:flex;gap:12px;flex-wrap:wrap;margin:25px 0">

<div style="background:#191919;padding:18px;border-left:4px solid #F5C518;flex:1;border-radius:6px">
<small style="color:#999">LONG</small>
<h1 style="color:#F5C518">{longs}</h1>
</div>

<div style="background:#191919;padding:18px;border-left:4px solid #F5C518;flex:1;border-radius:6px">
<small style="color:#999">SHORT</small>
<h1 style="color:#F5C518">{shorts}</h1>
</div>

<div style="background:#191919;padding:18px;border-left:4px solid #F5C518;flex:1;border-radius:6px">
<small style="color:#999">AGUARDAR</small>
<h1 style="color:#F5C518">{waiting}</h1>
</div>

</div>

<h3 style="color:#F5C518">MARKET OPPORTUNITIES</h3>

<style>
.vexa-table {{
width:100%;
border-collapse:collapse;
font-size:13px;
}}
.vexa-table th {{
background:#F5C518 !important;
color:#080808 !important;
padding:13px;
text-align:left;
}}
.vexa-table td {{
background:#151515 !important;
color:#eeeeee !important;
padding:13px;
border-bottom:1px solid #303030;
}}
.vexa-table tr:nth-child(even) td {{
background:#202020 !important;
}}
</style>

{html_table}

<p style="color:#666;font-size:10px;margin-top:25px">
VEXA SIGNALS | POWERED BY HYPERLIQUID
</p>

</div>
"""

display(HTML(html))


In [ ]:

# ==========================================
# VEXA SIGNALS — TRADE LEVELS
# ==========================================

trade_levels = []

for _, row in vexa_panel.iterrows():

    coin = row["Moeda"]
    signal = row["Sinal"]

    try:
        candles = get_candles(coin)
        candles = candles.sort_values("t").reset_index(drop=True)

        high = candles["h"]
        low = candles["l"]
        close = candles["c"]

        tr = pd.concat([
            high - low,
            (high - close.shift()).abs(),
            (low - close.shift()).abs()
        ], axis=1).max(axis=1)

        atr = tr.ewm(
            alpha=1/14,
            adjust=False,
            min_periods=14
        ).mean().iloc[-1]

        entry = float(close.iloc[-1])

        if signal == "LONG":
            stop = entry - (1.5 * atr)
            target = entry + (3 * atr)

        elif signal == "SHORT":
            stop = entry + (1.5 * atr)
            target = entry - (3 * atr)

        else:
            stop = np.nan
            target = np.nan

        trade_levels.append({
            "Moeda": coin,
            "Sinal": signal,
            "Entry": entry,
            "ATR": atr,
            "Stop Loss": stop,
            "Take Profit": target,
            "R/R": "1:2" if signal in ["LONG", "SHORT"] else "-"
        })

    except Exception as e:
        print(f"Erro em {coin}: {e}")

vexa_levels = pd.DataFrame(trade_levels)

print("VEXA — TRADE LEVELS")
display(vexa_levels.round(6))


In [ ]:

# ==========================================
# VEXA SIGNALS — PROFESSIONAL DASHBOARD
# ==========================================

from IPython.display import display, HTML
import pandas as pd

# Unir indicadores, score e níveis
dashboard = vexa_panel.merge(
    vexa_levels,
    on=["Moeda", "Sinal"],
    how="left",
    suffixes=("", "_trade")
)

# Selecionar sinais ativos
active = dashboard[
    dashboard["Sinal"].isin(["LONG", "SHORT"])
].copy()

# Contadores
longs = (dashboard["Sinal"] == "LONG").sum()
shorts = (dashboard["Sinal"] == "SHORT").sum()
waiting = (dashboard["Sinal"] == "AGUARDAR").sum()

# Formatar tabela completa
display_table = dashboard[
    ["Moeda", "Sinal", "Score", "Entry",
     "Stop Loss", "Take Profit", "R/R"]
].copy()

display_table["Score"] = (
    display_table["Score"].astype(str) + "/100"
)

for col in ["Entry", "Stop Loss", "Take Profit"]:
    display_table[col] = display_table[col].apply(
        lambda x: "-" if pd.isna(x) else f"{x:.6f}"
    )

html_table = display_table.to_html(
    index=False,
    border=0,
    classes="vexa-table"
)

# Gerar painel
html = f"""
<div style="
background:#080808;
color:#eeeeee;
padding:25px;
border-radius:14px;
font-family:Arial,sans-serif;
">

<div style="display:flex;justify-content:space-between;
align-items:center">

<h1 style="color:#F5C518;letter-spacing:3px;margin:0">
VEXA
</h1>

<span style="color:#F5C518;font-weight:bold">
SIGNALS
</span>
</div>

<p style="color:#888;font-size:11px;letter-spacing:2px">
CRYPTO MARKET INTELLIGENCE
</p>

<hr style="border:1px solid #F5C518">

<div style="display:flex;gap:12px;flex-wrap:wrap;margin:22px 0">

<div style="background:#191919;padding:16px;
border-left:4px solid #F5C518;flex:1;border-radius:6px">
<small style="color:#999">LONG</small>
<h1 style="color:#F5C518">{longs}</h1>
</div>

<div style="background:#191919;padding:16px;
border-left:4px solid #F5C518;flex:1;border-radius:6px">
<small style="color:#999">SHORT</small>
<h1 style="color:#F5C518">{shorts}</h1>
</div>

<div style="background:#191919;padding:16px;
border-left:4px solid #F5C518;flex:1;border-radius:6px">
<small style="color:#999">AGUARDAR</small>
<h1 style="color:#F5C518">{waiting}</h1>
</div>

</div>

<h3 style="color:#F5C518">ACTIVE OPPORTUNITIES</h3>

<p style="color:#999;font-size:12px">
Sinais identificados pelo motor de confluência
</p>

{active[["Moeda", "Sinal", "Score"]].to_html(
    index=False, border=0, classes="vexa-table"
) if not active.empty else
"<p style='color:#999'>Nenhum sinal ativo no momento.</p>"}

<h3 style="color:#F5C518;margin-top:28px">
TRADE LEVELS & MARKET WATCH
</h3>

<style>
.vexa-table {{
width:100%;
border-collapse:collapse;
font-size:12px;
}}
.vexa-table th {{
background:#F5C518 !important;
color:#080808 !important;
padding:12px;
text-align:left;
}}
.vexa-table td {{
background:#151515 !important;
color:#eeeeee !important;
padding:12px;
border-bottom:1px solid #303030;
}}
.vexa-table tr:nth-child(even) td {{
background:#202020 !important;
}}
</style>

{html_table}

<p style="color:#666;font-size:10px;margin-top:25px">
VEXA SIGNALS | HYPERLIQUID MARKET DATA
</p>

</div>
"""

display(HTML(html))


In [ ]:

# =====================================================
# VEXA SIGNALS 1.0 — UNIFIED MARKET ENGINE
# Hyperliquid | 15m | EMA | RSI | ADX | ATR | Volume
# =====================================================

import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime, timezone
from IPython.display import display

URL = "https://api.hyperliquid.xyz/info"

MIN_VOLUME = 30_000_000
MAX_MARKETS = 9
TIMEFRAME = "15m"
CANDLE_LIMIT = 260

# -----------------------------------------------------
# 1. MARKET SNAPSHOT
# -----------------------------------------------------

def get_market_snapshot():

    response = requests.post(
        URL,
        json={"type": "metaAndAssetCtxs"},
        timeout=20
    )
    response.raise_for_status()

    meta, contexts = response.json()

    markets = []

    for coin, ctx in zip(meta["universe"], contexts):

        markets.append({
            "Moeda": coin["name"],
            "Preço": float(ctx["markPx"]),
            "Volume_24h": float(ctx["dayNtlVlm"]),
            "Funding": float(ctx["funding"])
        })

    markets = pd.DataFrame(markets)

    markets = markets[
        markets["Volume_24h"] >= MIN_VOLUME
    ]

    return markets.sort_values(
        "Volume_24h", ascending=False
    ).head(MAX_MARKETS).reset_index(drop=True)


# -----------------------------------------------------
# 2. CLOSED CANDLES
# -----------------------------------------------------

def get_closed_candles(coin):

    now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)

    payload = {
        "type": "candleSnapshot",
        "req": {
            "coin": coin,
            "interval": TIMEFRAME,
            "startTime": now_ms - CANDLE_LIMIT * 15 * 60 * 1000,
            "endTime": now_ms
        }
    }

    response = requests.post(
        URL, json=payload, timeout=20
    )
    response.raise_for_status()

    data = response.json()

    if not data:
        return pd.DataFrame()

    candles = pd.DataFrame(data)

    for col in ["o", "h", "l", "c", "v"]:
        candles[col] = pd.to_numeric(
            candles[col], errors="coerce"
        )

    candles["T"] = pd.to_numeric(
        candles["T"], errors="coerce"
    )

    # T representa o encerramento do candle.
    # Excluir candles ainda abertos.
    candles = candles[
        candles["T"] < now_ms
    ].copy()

    candles = candles.sort_values("t")
    candles = candles.drop_duplicates("t")

    return candles.reset_index(drop=True)


# -----------------------------------------------------
# 3. INDICATORS + SIGNAL + ATR
# -----------------------------------------------------

def analyze_market(market):

    coin = market["Moeda"]

    candles = get_closed_candles(coin)

    if len(candles) < 220:
        return None

    d = candles.copy()

    close = d["c"]
    high = d["h"]
    low = d["l"]

    # EMA
    for period in [9, 21, 50, 200]:
        d[f"EMA{period}"] = close.ewm(
            span=period, adjust=False
        ).mean()

    # TRUE RANGE / ATR
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low - close.shift()).abs()
    ], axis=1).max(axis=1)

    d["ATR"] = tr.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    # RSI
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    avg_loss = loss.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)

    d["RSI"] = 100 - (100 / (1 + rs))
    d.loc[(avg_loss == 0) & (avg_gain > 0), "RSI"] = 100
    d.loc[(avg_gain == 0) & (avg_loss > 0), "RSI"] = 0

    # ADX
    up_move = high.diff()
    down_move = -low.diff()

    plus_dm = up_move.where(
        (up_move > down_move) & (up_move > 0), 0.0
    )

    minus_dm = down_move.where(
        (down_move > up_move) & (down_move > 0), 0.0
    )

    plus_di = 100 * plus_dm.ewm(
        alpha=1/14, adjust=False
    ).mean() / d["ATR"]

    minus_di = 100 * minus_dm.ewm(
        alpha=1/14, adjust=False
    ).mean() / d["ATR"]

    dx = 100 * (plus_di - minus_di).abs() / (
        plus_di + minus_di
    ).replace(0, np.nan)

    d["ADX"] = dx.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    # Relative volume
    d["VOL_MEDIA"] = d["v"].shift(1).rolling(20).mean()
    d["VOL_RATIO"] = d["v"] / d["VOL_MEDIA"]

    last = d.iloc[-1]

    required = [
        "EMA9", "EMA21", "EMA50", "EMA200",
        "ATR", "RSI", "ADX", "VOL_RATIO"
    ]

    if last[required].isna().any():
        return None

    price = float(last["c"])
    atr = float(last["ATR"])
    rsi = float(last["RSI"])
    adx = float(last["ADX"])
    volume_ratio = float(last["VOL_RATIO"])

    # Trend alignment
    long_trend = (
        last["EMA9"] > last["EMA21"] >
        last["EMA50"] > last["EMA200"]
    )

    short_trend = (
        last["EMA9"] < last["EMA21"] <
        last["EMA50"] < last["EMA200"]
    )

    # Signal conditions
    long_signal = (
        long_trend
        and price > last["EMA9"]
        and 52 <= rsi <= 70
        and adx >= 20
        and volume_ratio >= 1.2
    )

    short_signal = (
        short_trend
        and price < last["EMA9"]
        and 30 <= rsi <= 48
        and adx >= 20
        and volume_ratio >= 1.2
    )

    if long_signal:
        signal = "LONG"
    elif short_signal:
        signal = "SHORT"
    else:
        signal = "AGUARDAR"

    # Direction-specific confluence score
    score = 0
    reasons = []

    if signal != "AGUARDAR":

        score += 35
        reasons.append("Tendência confirmada")

        score += 20
        reasons.append("RSI favorável")

        score += min(25, adx / 30 * 25)
        reasons.append("ADX confirmado")

        score += min(20, volume_ratio / 2 * 20)
        reasons.append("Volume confirmado")

    else:
        reasons.append("Confluência incompleta")

    # Entry and risk levels from SAME candle
    entry = price
    stop = np.nan
    target = np.nan

    if signal == "LONG":
        stop = entry - 1.5 * atr
        target = entry + 3 * atr

    elif signal == "SHORT":
        stop = entry + 1.5 * atr
        target = entry - 3 * atr

    return {
        "Moeda": coin,
        "Sinal": signal,
        "Score": round(score),
        "Entry": entry,
        "Stop Loss": stop,
        "Take Profit": target,
        "ATR": atr,
        "RSI": rsi,
        "ADX": adx,
        "Volume x Média": volume_ratio,
        "Funding": market["Funding"],
        "Volume 24h": market["Volume_24h"],
        "Candle fechado UTC": datetime.fromtimestamp(
            int(last["T"]) / 1000,
            tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M"),
        "Justificativa": " | ".join(reasons)
    }


# -----------------------------------------------------
# 4. UNIFIED SCANNER
# -----------------------------------------------------

print("VEXA — INICIANDO ANÁLISE UNIFICADA")

market_snapshot = get_market_snapshot()

unified_results = []

for _, market in market_snapshot.iterrows():

    try:
        result = analyze_market(market)

        if result is not None:
            unified_results.append(result)
            print(f"{market['Moeda']}: {result['Sinal']}")

        else:
            print(f"{market['Moeda']}: histórico insuficiente")

    except Exception as error:
        print(f"{market['Moeda']}: erro - {error}")

    time.sleep(0.2)


# -----------------------------------------------------
# 5. FINAL RESULT
# -----------------------------------------------------

if not unified_results:
    raise RuntimeError("Nenhum mercado foi analisado com sucesso.")

vexa_live = pd.DataFrame(unified_results)

vexa_live["Prioridade"] = vexa_live["Sinal"].map({
    "LONG": 0,
    "SHORT": 0,
    "AGUARDAR": 1
})

vexa_live = vexa_live.sort_values(
    ["Prioridade", "Score"],
    ascending=[True, False]
).reset_index(drop=True)

longs = (vexa_live["Sinal"] == "LONG").sum()
shorts = (vexa_live["Sinal"] == "SHORT").sum()
waiting = (vexa_live["Sinal"] == "AGUARDAR").sum()

print("\n======================================")
print("          VEXA SIGNALS 1.0")
print("======================================")
print(f"Mercados analisados: {len(vexa_live)}")
print(f"LONG: {longs}")
print(f"SHORT: {shorts}")
print(f"AGUARDAR: {waiting}")
print("======================================")

display(
    vexa_live[
        [
            "Moeda", "Sinal", "Score",
            "Entry", "Stop Loss", "Take Profit",
            "RSI", "ADX", "Volume x Média"
        ]
    ].round(6)
)


In [ ]:

# =====================================================
# VEXA SIGNALS 1.0 — UNIFIED MARKET ENGINE
# Hyperliquid | 15m | EMA | RSI | ADX | ATR | Volume
# =====================================================

import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime, timezone
from IPython.display import display

URL = "https://api.hyperliquid.xyz/info"

MIN_VOLUME = 30_000_000
MAX_MARKETS = 9
TIMEFRAME = "15m"
CANDLE_LIMIT = 260

# -----------------------------------------------------
# 1. MARKET SNAPSHOT
# -----------------------------------------------------

def get_market_snapshot():

    response = requests.post(
        URL,
        json={"type": "metaAndAssetCtxs"},
        timeout=20
    )
    response.raise_for_status()

    meta, contexts = response.json()

    markets = []

    for coin, ctx in zip(meta["universe"], contexts):

        markets.append({
            "Moeda": coin["name"],
            "Preço": float(ctx["markPx"]),
            "Volume_24h": float(ctx["dayNtlVlm"]),
            "Funding": float(ctx["funding"])
        })

    markets = pd.DataFrame(markets)

    markets = markets[
        markets["Volume_24h"] >= MIN_VOLUME
    ]

    return markets.sort_values(
        "Volume_24h", ascending=False
    ).head(MAX_MARKETS).reset_index(drop=True)


# -----------------------------------------------------
# 2. CLOSED CANDLES
# -----------------------------------------------------

def get_closed_candles(coin):

    now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)

    payload = {
        "type": "candleSnapshot",
        "req": {
            "coin": coin,
            "interval": TIMEFRAME,
            "startTime": now_ms - CANDLE_LIMIT * 15 * 60 * 1000,
            "endTime": now_ms
        }
    }

    response = requests.post(
        URL, json=payload, timeout=20
    )
    response.raise_for_status()

    data = response.json()

    if not data:
        return pd.DataFrame()

    candles = pd.DataFrame(data)

    for col in ["o", "h", "l", "c", "v"]:
        candles[col] = pd.to_numeric(
            candles[col], errors="coerce"
        )

    candles["T"] = pd.to_numeric(
        candles["T"], errors="coerce"
    )

    # T representa o encerramento do candle.
    # Excluir candles ainda abertos.
    candles = candles[
        candles["T"] < now_ms
    ].copy()

    candles = candles.sort_values("t")
    candles = candles.drop_duplicates("t")

    return candles.reset_index(drop=True)


# -----------------------------------------------------
# 3. INDICATORS + SIGNAL + ATR
# -----------------------------------------------------

def analyze_market(market):

    coin = market["Moeda"]

    candles = get_closed_candles(coin)

    if len(candles) < 220:
        return None

    d = candles.copy()

    close = d["c"]
    high = d["h"]
    low = d["l"]

    # EMA
    for period in [9, 21, 50, 200]:
        d[f"EMA{period}"] = close.ewm(
            span=period, adjust=False
        ).mean()

    # TRUE RANGE / ATR
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low - close.shift()).abs()
    ], axis=1).max(axis=1)

    d["ATR"] = tr.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    # RSI
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    avg_loss = loss.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)

    d["RSI"] = 100 - (100 / (1 + rs))
    d.loc[(avg_loss == 0) & (avg_gain > 0), "RSI"] = 100
    d.loc[(avg_gain == 0) & (avg_loss > 0), "RSI"] = 0

    # ADX
    up_move = high.diff()
    down_move = -low.diff()

    plus_dm = up_move.where(
        (up_move > down_move) & (up_move > 0), 0.0
    )

    minus_dm = down_move.where(
        (down_move > up_move) & (down_move > 0), 0.0
    )

    plus_di = 100 * plus_dm.ewm(
        alpha=1/14, adjust=False
    ).mean() / d["ATR"]

    minus_di = 100 * minus_dm.ewm(
        alpha=1/14, adjust=False
    ).mean() / d["ATR"]

    dx = 100 * (plus_di - minus_di).abs() / (
        plus_di + minus_di
    ).replace(0, np.nan)

    d["ADX"] = dx.ewm(
        alpha=1/14, adjust=False, min_periods=14
    ).mean()

    # Relative volume
    d["VOL_MEDIA"] = d["v"].shift(1).rolling(20).mean()
    d["VOL_RATIO"] = d["v"] / d["VOL_MEDIA"]

    last = d.iloc[-1]

    required = [
        "EMA9", "EMA21", "EMA50", "EMA200",
        "ATR", "RSI", "ADX", "VOL_RATIO"
    ]

    if last[required].isna().any():
        return None

    price = float(last["c"])
    atr = float(last["ATR"])
    rsi = float(last["RSI"])
    adx = float(last["ADX"])
    volume_ratio = float(last["VOL_RATIO"])

    # Trend alignment
    long_trend = (
        last["EMA9"] > last["EMA21"] >
        last["EMA50"] > last["EMA200"]
    )

    short_trend = (
        last["EMA9"] < last["EMA21"] <
        last["EMA50"] < last["EMA200"]
    )

    # Signal conditions
    long_signal = (
        long_trend
        and price > last["EMA9"]
        and 52 <= rsi <= 70
        and adx >= 20
        and volume_ratio >= 1.2
    )

    short_signal = (
        short_trend
        and price < last["EMA9"]
        and 30 <= rsi <= 48
        and adx >= 20
        and volume_ratio >= 1.2
    )

    if long_signal:
        signal = "LONG"
    elif short_signal:
        signal = "SHORT"
    else:
        signal = "AGUARDAR"

    # Direction-specific confluence score
    score = 0
    reasons = []

    if signal != "AGUARDAR":

        score += 35
        reasons.append("Tendência confirmada")

        score += 20
        reasons.append("RSI favorável")

        score += min(25, adx / 30 * 25)
        reasons.append("ADX confirmado")

        score += min(20, volume_ratio / 2 * 20)
        reasons.append("Volume confirmado")

    else:
        reasons.append("Confluência incompleta")

    # Entry and risk levels from SAME candle
    entry = price
    stop = np.nan
    target = np.nan

    if signal == "LONG":
        stop = entry - 1.5 * atr
        target = entry + 3 * atr

    elif signal == "SHORT":
        stop = entry + 1.5 * atr
        target = entry - 3 * atr

    return {
        "Moeda": coin,
        "Sinal": signal,
        "Score": round(score),
        "Entry": entry,
        "Stop Loss": stop,
        "Take Profit": target,
        "ATR": atr,
        "RSI": rsi,
        "ADX": adx,
        "Volume x Média": volume_ratio,
        "Funding": market["Funding"],
        "Volume 24h": market["Volume_24h"],
        "Candle fechado UTC": datetime.fromtimestamp(
            int(last["T"]) / 1000,
            tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M"),
        "Justificativa": " | ".join(reasons)
    }


# -----------------------------------------------------
# 4. UNIFIED SCANNER
# -----------------------------------------------------

print("VEXA — INICIANDO ANÁLISE UNIFICADA")

market_snapshot = get_market_snapshot()

unified_results = []

for _, market in market_snapshot.iterrows():

    try:
        result = analyze_market(market)

        if result is not None:
            unified_results.append(result)
            print(f"{market['Moeda']}: {result['Sinal']}")

        else:
            print(f"{market['Moeda']}: histórico insuficiente")

    except Exception as error:
        print(f"{market['Moeda']}: erro - {error}")

    time.sleep(0.2)


# -----------------------------------------------------
# 5. FINAL RESULT
# -----------------------------------------------------

if not unified_results:
    raise RuntimeError("Nenhum mercado foi analisado com sucesso.")

vexa_live = pd.DataFrame(unified_results)

vexa_live["Prioridade"] = vexa_live["Sinal"].map({
    "LONG": 0,
    "SHORT": 0,
    "AGUARDAR": 1
})

vexa_live = vexa_live.sort_values(
    ["Prioridade", "Score"],
    ascending=[True, False]
).reset_index(drop=True)

longs = (vexa_live["Sinal"] == "LONG").sum()
shorts = (vexa_live["Sinal"] == "SHORT").sum()
waiting = (vexa_live["Sinal"] == "AGUARDAR").sum()

print("\n======================================")
print("          VEXA SIGNALS 1.0")
print("======================================")
print(f"Mercados analisados: {len(vexa_live)}")
print(f"LONG: {longs}")
print(f"SHORT: {shorts}")
print(f"AGUARDAR: {waiting}")
print("======================================")

display(
    vexa_live[
        [
            "Moeda", "Sinal", "Score",
            "Entry", "Stop Loss", "Take Profit",
            "RSI", "ADX", "Volume x Média"
        ]
    ].round(6)
)


In [ ]:

# ==========================================
# VEXA SIGNALS — LIVE ENGINE DASHBOARD
# ==========================================

from IPython.display import display, HTML

longs = (vexa_live["Sinal"] == "LONG").sum()
shorts = (vexa_live["Sinal"] == "SHORT").sum()
waiting = (vexa_live["Sinal"] == "AGUARDAR").sum()

active = vexa_live[
    vexa_live["Sinal"].isin(["LONG", "SHORT"])
].copy()

def fmt(value):
    return "-" if pd.isna(value) else f"{value:.6f}"

columns = [
    "Moeda", "Sinal", "Score", "Entry",
    "Stop Loss", "Take Profit", "RSI", "ADX"
]

table = vexa_live[columns].copy()

for col in ["Entry", "Stop Loss", "Take Profit"]:
    table[col] = table[col].apply(fmt)

table["Score"] = table["Score"].astype(str) + "/100"

html_table = table.to_html(
    index=False, border=0, classes="vexa-table"
)

active_table = active[
    ["Moeda", "Sinal", "Score", "Entry",
     "Stop Loss", "Take Profit"]
].copy()

for col in ["Entry", "Stop Loss", "Take Profit"]:
    active_table[col] = active_table[col].apply(fmt)

active_html = active_table.to_html(
    index=False, border=0, classes="vexa-table"
)

html = f"""
<div style="background:#080808;color:#eee;
padding:25px;border-radius:14px;font-family:Arial">

<h1 style="color:#F5C518;letter-spacing:3px;margin:0">
VEXA <span style="color:white">SIGNALS</span>
</h1>

<p style="color:#888;font-size:11px">
UNIFIED ENGINE 1.0 | HYPERLIQUID
</p>

<hr style="border:1px solid #F5C518">

<div style="display:flex;gap:12px;flex-wrap:wrap;margin:22px 0">

<div style="background:#191919;padding:18px;
border-left:4px solid #F5C518;flex:1">
LONG <h1 style="color:#F5C518">{longs}</h1>
</div>

<div style="background:#191919;padding:18px;
border-left:4px solid #F5C518;flex:1">
SHORT <h1 style="color:#F5C518">{shorts}</h1>
</div>

<div style="background:#191919;padding:18px;
border-left:4px solid #F5C518;flex:1">
AGUARDAR <h1 style="color:#F5C518">{waiting}</h1>
</div>

</div>

<h3 style="color:#F5C518">ACTIVE OPPORTUNITIES</h3>

{active_html if not active.empty else
"<p>Nenhum sinal ativo.</p>"}

<h3 style="color:#F5C518;margin-top:28px">
MARKET WATCH
</h3>

<style>
.vexa-table {{
width:100%;
border-collapse:collapse;
font-size:12px;
}}
.vexa-table th {{
background:#F5C518 !important;
color:#080808 !important;
padding:12px;
text-align:left;
}}
.vexa-table td {{
background:#151515 !important;
color:#eee !important;
padding:12px;
border-bottom:1px solid #333;
}}
.vexa-table tr:nth-child(even) td {{
background:#202020 !important;
}}
</style>

{html_table}

<p style="color:#777;font-size:10px;margin-top:22px">
VEXA SIGNALS | MARKET INTELLIGENCE
</p>
</div>
"""

display(HTML(html))


In [ ]:

# ==========================================
# VEXA SIGNALS 1.0 — LIVE MONITOR
# ==========================================

from IPython.display import display, HTML, clear_output
from datetime import datetime
import time
import pandas as pd
import numpy as np

REFRESH_SECONDS = 60

def vexa_refresh():

    markets = get_market_snapshot()
    results = []

    for _, market in markets.iterrows():
        try:
            result = analyze_market(market)

            if result is not None:
                # Usar mark price como referência atual
                entry = float(market["Preço"])
                atr = float(result["ATR"])

                result["Entry"] = entry

                if result["Sinal"] == "LONG":
                    result["Stop Loss"] = entry - 1.5 * atr
                    result["Take Profit"] = entry + 3 * atr

                elif result["Sinal"] == "SHORT":
                    result["Stop Loss"] = entry + 1.5 * atr
                    result["Take Profit"] = entry - 3 * atr

                results.append(result)

        except Exception as error:
            print(f"Erro em {market['Moeda']}: {error}")

        time.sleep(0.2)

    if not results:
        raise RuntimeError("Nenhum mercado analisado.")

    return pd.DataFrame(results)


def vexa_render(data):

    longs = (data["Sinal"] == "LONG").sum()
    shorts = (data["Sinal"] == "SHORT").sum()
    waiting = (data["Sinal"] == "AGUARDAR").sum()

    active = data[data["Sinal"].isin(["LONG", "SHORT"])]

    def fmt(x):
        return "-" if pd.isna(x) else f"{x:.6f}"

    cols = [
        "Moeda", "Sinal", "Score", "Entry",
        "Stop Loss", "Take Profit", "RSI", "ADX"
    ]

    table = data[cols].copy()

    for col in ["Entry", "Stop Loss", "Take Profit"]:
        table[col] = table[col].apply(fmt)

    table["Score"] = table["Score"].astype(str) + "/100"

    active_table = table[
        table["Sinal"].isin(["LONG", "SHORT"])
    ]

    html = f"""
    <div style="background:#080808;color:white;padding:25px;
    border-radius:14px;font-family:Arial">

    <h1 style="color:#F5C518;letter-spacing:3px">
    VEXA SIGNALS
    </h1>

    <p style="color:#999">HYPERLIQUID | LIVE MONITOR | 15 MIN</p>

    <p style="color:#F5C518">
    ÚLTIMA ATUALIZAÇÃO: {datetime.now().strftime("%H:%M:%S")}
    </p>

    <div style="display:flex;gap:12px;flex-wrap:wrap">

    <div style="background:#191919;padding:15px;flex:1">
    LONG <h1 style="color:#F5C518">{longs}</h1>
    </div>

    <div style="background:#191919;padding:15px;flex:1">
    SHORT <h1 style="color:#F5C518">{shorts}</h1>
    </div>

    <div style="background:#191919;padding:15px;flex:1">
    AGUARDAR <h1 style="color:#F5C518">{waiting}</h1>
    </div>

    </div>

    <h3 style="color:#F5C518;margin-top:25px">
    ACTIVE SIGNALS
    </h3>

    {active_table.to_html(index=False, border=0)}

    <h3 style="color:#F5C518;margin-top:25px">
    MARKET WATCH
    </h3>

    {table.to_html(index=False, border=0)}

    <style>
    table {{
        width:100%;
        border-collapse:collapse;
        font-size:12px;
    }}
    th {{
        background:#F5C518;
        color:#080808;
        padding:10px;
    }}
    td {{
        background:#191919;
        color:white;
        padding:10px;
        border-bottom:1px solid #333;
    }}
    </style>

    <p style="color:#777;font-size:10px">
    VEXA SIGNALS 1.0 | MANUAL TRADING ASSISTANT
    </p>
    </div>
    """

    display(HTML(html))


# MONITORAMENTO CONTÍNUO
print("VEXA SIGNALS — MONITORAMENTO INICIADO")
print("Atualização a cada 60 segundos.")
print("Para interromper, clique no botão STOP da célula.")

while True:
    try:
        vexa_live = vexa_refresh()

        clear_output(wait=True)
        vexa_render(vexa_live)

        time.sleep(REFRESH_SECONDS)

    except KeyboardInterrupt:
        print("VEXA SIGNALS — MONITORAMENTO ENCERRADO")
        break

    except Exception as error:
        clear_output(wait=True)
        print("VEXA — ERRO DE ATUALIZAÇÃO:", error)
        print("Nova tentativa em 60 segundos.")
        time.sleep(REFRESH_SECONDS)
